We gather data daily, so each source is referenced daily.

In [0]:
from pyspark.sql import functions as F


# Cut days at midnight
spark.conf.set("spark.sql.session.timeZone", "Europe/Brussels")

# Collect data
silver = spark.table("workspace.energy.silver_generation")

# each row cover one hour
HOURS_PER_ROW = 1.0

# we add a column with only the date
gold_daily = silver.withColumn("date", F.to_date("timestamp"))

gold_daily = gold_daily.withColumn("mwh", F.col("mw") * F.lit(HOURS_PER_ROW))

gold_daily = gold_daily.groupBy("date", "source", "category").agg(
    F.sum("mwh").alias("mwh"),
    F.count("*").alias("nb_hours"),
    F.max(F.col("is_imputed").cast("int")).alias("has_imputed")
)

gold_daily = gold_daily.withColumn(
    "is_generation",
    F.col("category").isin("nuclear", "fossil", "renewable", "other"),
)
# overwrite = modify, add data, true = modify column
gold_daily.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.energy.gold_daily_by_source")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Production only (step 2)
daily = spark.table("workspace.energy.gold_daily_by_source").filter(F.col("is_generation"))

# ---------- Monthly mix ----------
monthly = daily.withColumn("month", F.trunc("date", "month"))   # 2025-03-17 -> 2025-03-01

monthly = monthly.groupBy("month", "source", "category").agg(
    F.sum("mwh").alias("mwh"),
    F.max("has_imputed").alias("has_imputed"),
)

# Total production of the month, repeated on every row of that month
month_window = Window.partitionBy("month")
monthly = monthly.withColumn("total_mwh", F.sum("mwh").over(month_window))
monthly = monthly.withColumn("share_pct", F.col("mwh") / F.col("total_mwh") * 100)

monthly.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.energy.gold_monthly_mix")

# ---------- Yearly mix ----------
yearly = daily.withColumn("year", F.year("date"))

yearly = yearly.groupBy("year", "source", "category").agg(
    F.sum("mwh").alias("mwh"),
    F.max("has_imputed").alias("has_imputed"),
)

year_window = Window.partitionBy("year")
yearly = yearly.withColumn("total_mwh", F.sum("mwh").over(year_window))
yearly = yearly.withColumn("share_pct", F.col("mwh") / F.col("total_mwh") * 100)

yearly.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.energy.gold_yearly_mix")

print("Monthly and yearly mix saved ✅")

In [0]:
%sql
SELECT year, ROUND(SUM(share_pct), 2) AS total_share
FROM workspace.energy.gold_yearly_mix
GROUP BY year
ORDER BY year;

In [0]:
%sql
SELECT year, category, ROUND(SUM(share_pct), 1) AS share_pct
FROM workspace.energy.gold_yearly_mix
WHERE year IN (2024, 2025)
GROUP BY year, category
ORDER BY year, share_pct DESC;

Step 4

In [0]:
bronze = spark.table("workspace.energy.bronze_public_power")

balance = bronze.withColumn("date", F.to_date("timestamp")).groupBy("date").agg(
    F.sum(F.col("load") * HOURS_PER_ROW).alias("load_mwh"),
    F.sum(F.col("cross_border_electricity_trading") * HOURS_PER_ROW).alias("net_import_mwh"),
)

generation = (
    spark.table("workspace.energy.gold_daily_by_source")
    .filter(F.col("is_generation"))
    .groupBy("date")
    .agg(
        F.sum("mwh").alias("generation_mwh"),
        F.sum(F.when(F.col("category") == "renewable", F.col("mwh")).otherwise(0)).alias("renewable_mwh"),
    )
)

balance = balance.join(generation, on="date", how="left")
balance = balance.withColumn("renewable_share_of_load_pct", F.col("renewable_mwh") / F.col("load_mwh") * 100)
balance = balance.withColumn("net_import_share_of_load_pct", F.col("net_import_mwh") / F.col("load_mwh") * 100)

balance.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.energy.gold_daily_balance")
print("Gold daily balance saved ✅")

In [0]:
%sql
SELECT g.year, g.ours_pct, b.energy_charts_pct
FROM (
  SELECT YEAR(date) AS year,
         ROUND(SUM(renewable_mwh) / SUM(load_mwh) * 100, 1) AS ours_pct
  FROM workspace.energy.gold_daily_balance
  GROUP BY 1
) g
JOIN (
  SELECT YEAR(`timestamp`) AS year,
         ROUND(SUM(renewable_share_of_load * load) / SUM(load), 1) AS energy_charts_pct
  FROM workspace.energy.bronze_public_power
  GROUP BY 1
) b ON g.year = b.year
ORDER BY 1;

Step 5

In [0]:
profile = spark.table("workspace.energy.silver_generation")

profile = profile.withColumn("year", F.year("timestamp"))
profile = profile.withColumn("hour", F.hour("timestamp"))
profile = profile.withColumn("month_num", F.month("timestamp"))

season = (
    F.when(F.col("month_num").isin(12, 1, 2), "winter")
    .when(F.col("month_num").isin(3, 4, 5), "spring")
    .when(F.col("month_num").isin(6, 7, 8), "summer")
    .otherwise("autumn")
)
profile = profile.withColumn("season", season)

profile = profile.groupBy("year", "season", "hour", "source", "category").agg(
    F.avg("mw").alias("avg_mw")
)

profile.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.energy.gold_hourly_profile")
print("Gold hourly profile saved ✅")

In [0]:
%sql
COMMENT ON TABLE workspace.energy.gold_daily_by_source IS
  'Daily energy (MWh) per source, Belgian days. is_generation = false for storage and cross-border exchange. has_imputed = 1 if any hourly value was imputed.';

COMMENT ON TABLE workspace.energy.gold_monthly_mix IS
  'Monthly generation per source and share of total generation (%). Production sources only.';

COMMENT ON TABLE workspace.energy.gold_yearly_mix IS
  'Yearly generation per source and share of total generation (%). Production sources only.';

COMMENT ON TABLE workspace.energy.gold_daily_balance IS
  'Daily consumption, generation, renewable generation and net imports (MWh), with renewable and import shares of consumption.';

COMMENT ON TABLE workspace.energy.gold_hourly_profile IS
  'Average power (MW) per source by year, season and hour of day (Belgian time).';

In [0]:
%sql
SELECT hour, ROUND(avg_mw, 0) AS solar_mw
FROM workspace.energy.gold_hourly_profile
WHERE year = 2025 AND season = 'summer' AND source = 'solar'
ORDER BY hour;

In [0]:
%sql
SELECT DISTINCT is_generation, category, source
FROM workspace.energy.gold_daily_by_source
ORDER BY is_generation DESC, category, source;

In [0]:
%sql
SELECT YEAR(date) AS year,
       ROUND(SUM(mwh) / 1000000, 1) AS total_generation_twh
FROM workspace.energy.gold_daily_by_source
WHERE is_generation = true
  AND YEAR(date) IN (2024, 2025)
GROUP BY 1
ORDER BY 1;

In [0]:
%sql 
SELECT *
FROM workspace.energy.gold_daily_by_source
ORDER BY date
LIMIT 20;

Quick display

In [0]:
%sql
SELECT COUNT(*) as number_of_rows
FROM workspace.energy.gold_daily_by_source;

SELECT *
FROM workspace.energy.gold_daily_by_source
LIMIT 20;



SET TIME ZONE 'Europe/Brussels';

SELECT *
FROM workspace.energy.silver_generation
WHERE `timestamp` = '2026-05-04 12:00:00'
ORDER BY category, source;



SELECT COUNT(*)               AS nb_rows,
       COUNT(DISTINCT date)   AS nb_days,
       COUNT(DISTINCT source) AS nb_sources
FROM workspace.energy.gold_daily_by_source;

Comparison with Elia

In [0]:
%sql
SELECT YEAR(date) AS year,
       source,
       ROUND(SUM(mwh) / 1000000, 2) AS twh
FROM workspace.energy.gold_daily_by_source
WHERE YEAR(date) IN (2024, 2025)
  AND source IN ('nuclear', 'fossil_gas', 'solar', 'wind_onshore', 'wind_offshore')
GROUP BY 1, 2
ORDER BY 1, 2;

In [0]:
%sql
SET TIME ZONE 'Europe/Brussels';

SELECT DATE_TRUNC('day', `timestamp`) AS day,
       COUNT(*) AS null_hours
FROM workspace.energy.bronze_public_power
WHERE fossil_oil IS NULL
GROUP BY 1
ORDER BY 1;